# Graph Neural Networks (GNNs)

Goal:

Understand how to model structured data such as molecules.

Key Idea:

Data is not a table or sequence

Data is a graph:

    nodes + edges + relationships

## Why Standard ML Fails for Molecules

### Classical ML assumption:

data = table

rows --> samples  
columns --> features  

### But molecules are not tables:

They are graphs:

- atoms --> nodes  
- bonds --> edges  

Example:

    C — C = O

### Problem

We lose structure if we flatten into features

### Key Insight

We need a model that understands:

    relationships between elements


## Manual Solution (Intuition)

If you analyze a molecule:

You look at:

- neighboring atoms  
- bond types  
- local structure  

Example:

Carbon connected to Oxygen behaves differently

### Problem

Manual feature engineering:

- complex
- incomplete
- not scalable

### Solution

Let the model learn structure automatically

## Graph Representation
### Nodes

represent entities (atoms)

### Edges

represent relationships (bonds)

### Adjacency Matrix

defines connections:

A[i, j] = 1 --> nodes connected

### Key Insight

Graph = structure + features

In [ ]:
# SIMPLE GRAPH REPRESENTATION

import torch

# Example graph:
# 3 nodes

# node features
X = torch.tensor([
    [1.0],  # node 1
    [2.0],  # node 2
    [3.0]   # node 3
])

# adjacency matrix (connections)
A = torch.tensor([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
], dtype=torch.float32)

print("Node features:\n", X)
print("Adjacency matrix:\n", A)

## Graph Neural Network Idea

Each node updates its information by:

    looking at its neighbors

### Message Passing

At each step:

    new_state = aggregate(neighbors)

### Example

Node 2 sees:

- node 1
- node 3

--> updates its representation

### Key Insight

Nodes learn from their local environment

## GNN Update Rule

At each layer:

    h_i^(t+1) = f(h_i^(t), neighbors)

Simplified:

    H' = A X W

Where:

- X = node features  
- A = adjacency matrix  
- W = learnable weights  

### Interpretation

- multiply by A --> gather neighbor info
- multiply by W --> transform features

### Key Insight

GNN = repeated neighborhood aggregation

In [ ]:
# SIMPLE GNN LAYER

import torch.nn as nn

class SimpleGNNLayer(nn.Module):
    
    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, X, A):
        
        # aggregate neighbor information
        X_agg = torch.matmul(A, X)
        
        # transform features
        return self.linear(X_agg)

## Model Structure

Step 1:

update node features using neighbors

Step 2:

repeat (multi-layer propagation)

Step 3:

aggregate graph representation

### Key Insight

Local --> global representation

In [ ]:
# SIMPLE GNN MODEL

class SimpleGNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.gnn1 = SimpleGNNLayer(1, 8)
        self.gnn2 = SimpleGNNLayer(8, 4)
        
        self.fc = nn.Linear(4, 1)

    def forward(self, X, A):
        
        X = torch.relu(self.gnn1(X, A))
        X = torch.relu(self.gnn2(X, A))
        
        # global pooling (sum over nodes)
        X_graph = X.mean(dim=0)
        
        return self.fc(X_graph)

model = SimpleGNN()

In [ ]:
# TRAINING

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# fake target
y = torch.tensor([1.0])

for epoch in range(50):
    
    optimizer.zero_grad()
    
    output = model(X, A)
    
    loss = criterion(output.squeeze(), y)
    
    loss.backward()
    
    optimizer.step()
    
    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

## What is the GNN learning?

Each node learns:

    representation of its local environment

After layers:

    nodes contain global context

Graph embedding:

    summarizes entire structure


### Example (molecules)

Node (Carbon) learns:

- connected atoms
- bond patterns

### Key Insight

GNN = structure-aware feature learning

## Key Properties of GNNs

### Permutation Invariance

Order of nodes does NOT matter

### Locality

Information spreads through neighbors

### Depth

More layers --> larger neighborhood

### Risk

Too many layers:

    over-smoothing (all nodes look similar)

### Key Insight

GNN balances:

    locality vs global structure

## Molecular Interpretation
Molecule:

    graph of atoms


GNN learns:

- atom environments
- functional groups
- structural patterns

Applications:

- property prediction
- toxicity prediction
- activity prediction

### Key Insight

GNN = natural model for molecular structure